# 01 — Frozen backbone feature smoke test
Run notebook 00 first and keep the same Colab kernel connected. This independently tests both frozen backbones on one COD image.

In [17]:
import os
PROJECT_DIR = '/content/cod-ssl'
SAMPLE_IMAGE = '/content/cod_ssl_sample_image.png'
os.environ.setdefault('DINOV3_REPO_DIR','/content/third_party/dinov3')
os.environ.setdefault('DINOV3_WEIGHTS','/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth')
os.environ.setdefault('VJEPA2_REPO_DIR','/content/third_party/vjepa2')
os.environ.setdefault('VJEPA21_WEIGHTS','/content/drive/MyDrive/cod-ssl/checkpoints/vjepa2_1_vitb_dist_vitG_384.pt')

'/content/drive/MyDrive/cod-ssl/checkpoints/vjepa2_1_vitb_dist_vitG_384.pt'

In [18]:
from pathlib import Path
import torch
if not torch.cuda.is_available(): raise RuntimeError('Connect to a GPU-backed Colab kernel.')
required=[PROJECT_DIR,SAMPLE_IMAGE,os.environ['DINOV3_REPO_DIR'],os.environ['DINOV3_WEIGHTS'],os.environ['VJEPA2_REPO_DIR'],os.environ['VJEPA21_WEIGHTS']]
missing=[path for path in required if not Path(path).exists()]
if missing: raise FileNotFoundError('Missing paths:\n'+'\n'.join(missing))
print('GPU:',torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-40GB


In [21]:
%cd /content/cod-ssl
!git pull
!pip install -e ".[dev,notebooks]"

/content/cod-ssl
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 29.55 KiB | 2.46 MiB/s, done.
From https://github.com/papanag/cod-ssl
   838d1e1..20bb501  main       -> origin/main
Updating 838d1e1..20bb501
Fast-forward
 pyproject.toml |  20 ++-
 uv.lock        | 450 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 469 insertions(+), 1 deletion(-)
Obtaining file:///content/cod-ssl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [22]:
# DINOv3 layers 2, 5, 8, 11
import subprocess, sys
result = subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/inspect_backbone.py','--backbone','dinov3_vitb16','--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'DINOv3 inspection failed with exit code {result.returncode}')

Downloading: "file:///content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vitb16.pth

  0%|          | 0.00/327M [00:00<?, ?B/s]
 34%|███▍      | 112M/327M [00:00<00:00, 1.18GB/s]
 69%|██████▉   | 226M/327M [00:00<00:00, 1.19GB/s]
100%|██████████| 327M/327M [00:00<00:00, 1.19GB/s]
{
  "model": "dinov3_vitb16",
  "repo_commit": "6876159a11b4df116f30f667f8c9888617df0751",
  "checkpoint": "/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth",
  "checkpoint_sha256": "73cec8be7427c8655ceced13ce62f6e20a1fa90d1b4d4a550df17a1144081a7c",
  "total_parameters": 85669632,
  "trainable_parameters": 0,
  "input_shape": [
    1,
    3,
    384,
    384
  ],
  "feature_shapes": [
    [
      1,
      768,
      24,
      24
    ],
    [
      1,
      768,
      24,
      24
    ],
    [
      1,
      768,
      24,
      24
    ],
    [
      1,
      768,
      24,
      24
    ]
  ],
  "dtype": "torch.float32",
  "mean_forward_ms": 265

In [23]:
# Release allocator state before the second process.
import gc
gc.collect(); torch.cuda.empty_cache()

In [24]:
# V-JEPA 2.1 native one-frame pathway; predictor unused.
result = subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/inspect_backbone.py','--backbone','vjepa21_vitb16','--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'V-JEPA inspection failed with exit code {result.returncode}')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/lib/python3.13/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
{
  "model": "vjepa21_vitb16",
  "repo_commit": "204698b45b3712590f06245fbfba32d3be539812",
  "checkpoint": "/content/drive/MyDrive/cod-ssl/checkpoints/vjepa2_1_vitb_dist_vitG_384.pt",
  "checkpoint_sha256": "848a77c33cc9e6649ed2119c9bea1e2c569bcdab9539ff3e7c02ccc2959ddf4d",
  "total_parameters": 86833152,
  "trainable_parameters": 0,
  "input_shape": [
    1,
    3,
    384,
    384
  ],
  "feature_shapes": [
    [
      1,
      7

Each JSON report must show four feature maps with spatial size `24 × 24` and zero trainable backbone parameters.